In [1]:

import scanpy as sc
import os

file_id = "clean_testing_file_1.h5ad"
UPLOAD_DIR = "../../../persistent01"

file_path = os.path.join(UPLOAD_DIR, file_id)

adata = sc.read_h5ad(file_path)

adata


AnnData object with n_obs × n_vars = 177170 × 44643
    obs: 'run_id', 'sample', 'region', 'run_batch', 'time_to_surgery_old', 'sex_old', 't21_old', 'disease_old', 'donor', 'enrichment', 'tissue', 'scrublet_score', 'scrublet_leiden', 'cluster_scrublet_score', 'doublet_pval', 'doublet_bh_pval', 'batch', 'percent_mito', 'percent_ribo', 'n_counts', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'QC', 'droplet', 'S_score', 'G2M_score', 'phase', '_scvi_batch', '_scvi_labels', 'leiden_scVI', 'doublet_cls', 'localAnnotation', 'globalAnnotation', 'region_final', 'sex_final', 'time_to_surgery_final', 't21_final', 'localAnnotation_region_final', 'tissue_final', 'localAnnotation1', 'globalAnnotation1', 'batch_key', 'leiden_1_0_test_2905_1'
    var: 'gene_ids-0-0', 'n_cells-0-0', 'mt-0-0', 'ribo-0-0', 'hb-0-0', 'n_cells_by_counts-0-0', 'mean_counts-0-0', 'pct_dropout_by_counts-0-0', 'to

In [ ]:

adata.file.close()

In [4]:

for i in adata.uns_keys():
  if ("dotplot" in i):
    del adata.uns[i]

# del adata.uns["dotplot_stats_leiden_1_0_test_2905_1"]

adata.uns_keys()

# adata

['X_umap_test_2905_1',
 'dendrogram_leiden_1_0_test_2905_1',
 'leiden_1_0_test_2905_1',
 'pca',
 'rank_genes_groups_leiden_1_0_test_2905_1',
 'test_2905_1']

In [ ]:

new_file_id = "clean_testing_file_1.h5ad"
new_file_path = os.path.join(UPLOAD_DIR, new_file_id)

sc.write(new_file_path, adata)


In [ ]:


uns_key = "leiden_1_0_test_2905_1"


In [ ]:

adata.uns["dotplot_stats_leiden_1_0_test_2905_1"]

In [ ]:

rgg_uns_key = "rank_genes_groups_" + uns_key

adata.uns[rgg_uns_key]

In [ ]:

fig = sc.pl.rank_genes_groups_dotplot(
  adata,
  key=rgg_uns_key,
  n_genes=1,
  # values_to_plot="logfoldchanges",
  # cmap="bwr",
  # vmin=-4,
  # vmax=4,
  return_fig=True
)

fig.show()

In [ ]:

# top_genes = list(adata.var_names)

n_groups = 4

top_genes = set()
for group in adata.uns["rank_genes_groups_" + uns_key]["names"].dtype.names:
  top_genes.update(adata.uns["rank_genes_groups_" + uns_key]["names"][group][:n_groups])  # top_n = 4

print(len(top_genes), top_genes)


In [ ]:

# sc.tl.dendrogram(adata, groupby=uns_key)
sc.pl.dendrogram(adata, groupby=uns_key)


In [ ]:

from scipy.cluster.hierarchy import linkage, to_tree
import json

def build_dendrogram_tree(linkage_matrix, labels: list[str]):
  
  tree, nodes = to_tree(linkage_matrix, rd=True)
  
  def add_node(node):
    if node.is_leaf():
      return {"name": node.id}
    else:
      return {
        "name": None,
        "children": [add_node(node.left), add_node(node.right)],
        "distance": node.dist  # Optional: include distance info
      }

  return add_node(tree)

dendrogram_data = adata.uns[f"dendrogram_{uns_key}"]
dendrogram_order = dendrogram_data["categories_ordered"]
  
dendrogram_tree = build_dendrogram_tree(dendrogram_data["linkage"], dendrogram_order)

dendrogram_tree

with open("d3_dendrogram_2805_1.json", "w") as f:
  json.dump(dendrogram_tree, f, indent=2)

In [ ]:

adata.var_names[10:]

In [ ]:

# dp = sc.pl.DotPlot(adata, groupby=uns_key, var_names=adata.var_names[10:], )

# dp.show()

fig

# adata.file.close()

mean_expr_df = fig.dot_color_df # mean expression
frac_expr_df = fig.dot_size_df # fraction expression

print(mean_expr_df)
# print(mean_expr_df.columns.values)


In [ ]:

dupe_cols = mean_expr_df.columns[mean_expr_df.columns.duplicated()]

for i in dupe_cols:
  print(len(mean_expr_df[i].columns))

print(mean_expr_df.columns)
# dupe_cols

In [ ]:

# mean_expr_df.columns[mean_expr_df.columns.duplicated()].unique()

import pandas as pd

# Example: assume `mean_expr_df` is your DataFrame

# 1. Get a list of duplicate column names
dupe_cols = mean_expr_df.columns[mean_expr_df.columns.duplicated()].unique()

res = {
  
}

# 2. For each duplicate column name, find all its appearances and compare
for col in dupe_cols:
    col_indices = [i for i, c in enumerate(mean_expr_df.columns) if c == col]
    print(f"\nColumn '{col}' appears at positions {col_indices}")
    
    if not (col in res.keys()):
      res[col] = 0
    
    # Compare all duplicate columns pairwise
    base_series = mean_expr_df.iloc[:, col_indices[0]]
    for idx in col_indices[1:]:
        comp_series = mean_expr_df.iloc[:, idx]
        equal = base_series.equals(comp_series)
        if not equal:
          diff = base_series != comp_series
          print(f"❌ Difference found at rows: {diff[diff].index.tolist()}")
        else:
          res[col] += 1
        #     print(f"✅ Column at index {idx} is identical to the first occurrence.")


for i in res.keys():
  
  print(i, res[i])


In [ ]:

mean_expr_df_no_dup = mean_expr_df.loc[:, ~mean_expr_df.T.duplicated()]
frac_expr_df_no_dup = frac_expr_df.loc[:, ~frac_expr_df.T.duplicated()]

frac_expr_df


In [ ]:

# mean_expr_df = mean_expr_df.loc[dendrogram_order]
# frac_expr_df = frac_expr_df.loc[dendrogram_order]

# Melt mean and frac expr
mean_expr_long = mean_expr_df_no_dup.reset_index().melt(id_vars="index", var_name="gene", value_name="mean_expr").rename(columns={"index": "group"})
frac_expr_long = frac_expr_df_no_dup.reset_index().melt(id_vars="index", var_name="gene", value_name="frac_expr").rename(columns={"index": "group"})

merged_expr = mean_expr_long.merge(frac_expr_long, on=["gene", "group"])

print(merged_expr)
print(len(mean_expr_df_no_dup.columns))


In [ ]:

group_0 = merged_expr.loc[merged_expr["group"] == "1"]

len(group_0)

for i in merged_expr["group"]:
  gr0_len = len(merged_expr.loc[merged_expr["group"] == i])
  print(i, gr0_len)


In [ ]:

merged_expr.sort_values(by="group")

In [ ]:

genes = merged_expr["gene"].unique()

len(genes)

# for g in genes:
#   x = len(merged_expr["group"].loc[merged_expr["gene"] == g])
#   if (x < 120):
#     print(x)
  # print(len(merged_expr["group"].loc[merged_expr["gene"] == g]))

# genes

# len()

# i = 0
# for g in genes:
#   print(g)
#   # i += 1
#   # if (i == 10):
#   #   break
  


In [ ]:

mean_expr_df

mean_expr_df

for i in range(10):
  print(mean_expr_df[i])

# for group in rgg["names"].dtype.names[:10]:
  
  
  # mean_expr_df[]
#   print(f"{group}: {list(rgg['names'][group][:10])}")

In [ ]:
# Rank genes groups info
rgg = adata.uns[rgg_uns_key]
groups = rgg["names"].dtype.names

groups


In [ ]:

# print(rgg["names"]["0"], len(rgg["names"]["0"]))
# print(rgg["names"][0], len(rgg["names"][0]))

for group in rgg["names"].dtype.names[:10]:
  print(f"{group}: {list(rgg['names'][group][:10])}")

In [ ]:

pvals = []
logfcs = []
ranks = []

for index, row in merged_expr.iterrows():
  
  gene, group = row["gene"], row["group"]
  
  try:
    # print(gene, group)
    gene_list = rgg["names"][group]
    
    index = list(gene_list).index(gene)
    
    # print(gene_list, index)
    
    pval = rgg["pvals"][group][index]
    logfc = rgg["logfoldchanges"][group][index]
    rank = index
    
  except ValueError:
    pval = None
    logfc = None
    rank = None
    
  pvals.append(pval)
  logfcs.append(logfc)
  ranks.append(rank)

pvals


In [ ]:

merged_expr["pval"] = pvals
merged_expr["logfoldchange"] = logfcs
merged_expr["rank"] = ranks

# merged_expr.loc[merged_expr["logfoldchange"].isnull()]
merged_expr

In [ ]:

ranks

In [ ]:

import pandas as pd
pd.set_option("display.max_rows", 120)

dendro_order = adata.uns[f"dendrogram_{uns_key}"]["categories_ordered"]
dendro_map = {group: i for i, group in enumerate(dendro_order)}

N = 2

sorted_df = (
  merged_expr
    .assign(
      dendro_order=pd.Categorical(merged_expr["group"] ,categories=dendro_order, ordered=True)
      # dendro_order=merged_expr["group"].map(dendro_map)
    )
    .sort_values(by=["dendro_order", "rank"])
)

sorted_df = (
  sorted_df
    .assign(
      within_group_rank=sorted_df.groupby("group").cumcount(),
      batch=lambda df: df["within_group_rank"] // N
    )
    .sort_values(
      by=[
        "batch",
        "dendro_order",
      ],
      ascending=[
        True,
        True,
      ]
    )
)

(
  sorted_df.head(120)
  # .loc[sorted_df["group"] == "37"]
)


# # merged_expr.loc[merged_expr["group"] == "1"].head(30)
# # merged_expr.sort_values(by=["rank", "group"]).head(120)
# merged_expr["dendro_order"] = pd.Categorical(
#   merged_expr["group"],
#   categories=dendro_order,
#   ordered=True
# )

# merged_expr.sort_values(by=["dendro_order", "rank"], ascending=True).head(50)


In [ ]:

merged_expr["group_order"] = merged_expr.groupby("group").cumcount()

merged_expr.sort_values(by=["group_order", "dendro_order", "rank"]).head(120)



In [ ]:

rgg_vals = adata.uns[rgg_uns_key]

rgg_groups = rgg_vals["names"].dtype.names

rgg_groups

# rgg_vals.keys()
len(rgg_vals["logfoldchanges"][rgg_groups[5]])
len(rgg_groups)